# Silver local - leitura da camada Bronze

In [ ]:
import os
from pathlib import Path
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

java_home = Path("/usr/lib/jvm/java-17-openjdk-amd64")
if java_home.exists():
    os.environ["JAVA_HOME"] = str(java_home)
    os.environ["PATH"] = str(java_home / "bin") + os.pathsep + os.environ.get("PATH", "")
else:
    raise RuntimeError("JDK 17 não encontrado em /usr/lib/jvm/java-17-openjdk-amd64. Instale o OpenJDK 17 para rodar o Spark localmente.")

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / "landing" / "dados").exists():
        project_root = candidate
        break

workspace_dir = (project_root / "spark-warehouse").resolve()
caminho_bronze = (project_root / "bronze").resolve()
caminho_silver = (project_root / "silver").resolve()

os.makedirs(caminho_silver, exist_ok=True)
os.makedirs(workspace_dir, exist_ok=True)

try:
    current = SparkSession.getActiveSession()
    if current is not None:
        current.stop()
except Exception:
    pass

builder = (
    SparkSession.builder
    .master("local[*]")
    .appName("Projeto_Medalhao_Local_Silver")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", str(workspace_dir))
)

print(f"Silver: {caminho_silver}")
spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f"Bronze: {caminho_bronze}")
print(f"Project root: {project_root}")

# Leitura da camada Bronze em Delta local
print("Dados da camada Bronze carregados com sucesso!")

df_subjects = spark.read.format("delta").load(str(caminho_bronze / "subjects"))
df_classes = spark.read.format("delta").load(str(caminho_bronze / "classes"))
df_grades = spark.read.format("delta").load(str(caminho_bronze / "grades"))
df_students = spark.read.format("delta").load(str(caminho_bronze / "students"))
df_teacher = spark.read.format("delta").load(str(caminho_bronze / "teacher"))

IndentationError: unexpected indent (2840805148.py, line 35)

## Adicionar coluna de auditoria da camada Silver

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

# Inclusão de metadados da camada Silver
df_teacher = df_teacher.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("teacher"))
df_classes = df_classes.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("classes"))
df_students = df_students.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("students"))
df_subjects = df_subjects.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("subjects"))
df_grades = df_grades.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("grades"))


## Funções de limpeza e padronização de colunas

In [ ]:
def apply_name_rules(colname: str) -> str:
    n = colname.strip().replace("\ufeff", "").upper().replace(" ", "_")
    n = n.replace("ID_", "IDENTIFIER_")
    n = n.replace("ID", "IDENTIFIER")
    return n


def safe_drop(df, cols):
    existing = set(df.columns)
    to_drop = [c for c in cols if c in existing]
    return df.drop(*to_drop) if to_drop else df


## Persistir tabelas Silver em Delta local

In [ ]:
dataframes = {
    "teacher": df_teacher,
    "classes": df_classes,
    "students": df_students,
    "subjects": df_subjects,
    "grades": df_grades,
}

for nome_tabela, curr_df in dataframes.items():
    new_cols = [apply_name_rules(c) for c in curr_df.columns]
    curr_df = curr_df.toDF(*new_cols)
    curr_df = safe_drop(curr_df, ["DATA_HORA_BRONZE", "NOME_ARQUIVO"])
    curr_df.write.format("delta").mode("overwrite").save(str(caminho_silver / nome_tabela))
    print(f"Tabela Silver salva: {nome_tabela}")


26/10/21 00:20:39 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Tabela Silver salva: teacher


Tabela Silver salva: classes


Tabela Silver salva: students


Tabela Silver salva: subjects


Tabela Silver salva: grades


## Validação da camada Silver

In [ ]:
import os
print("Arquivos Silver:")
print(sorted(os.listdir(caminho_silver)))


['classes', 'teacher', 'students', 'subjects', 'grades']
